[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/en_clase/S15_01_seguimiento_experimentos_en_clase.ipynb)

# S15 · 01 — Seguimiento de experimentos y reproducibilidad · versión en clase

⏱ 50 min en clase · Sprint 15 · Proyecto: modelado y experimentos (píldora de apoyo)

**Contexto TurisData.** En el proyecto final vuestro equipo probará muchos modelos y muchos ajustes. Al tercer día nadie recuerda *"¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?"*. La rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier equipo. Usamos las reservas de TurisData como datos de práctica; en el proyecto usaréis los vuestros.

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. El ajuste de hiperparámetros registrado (ejercicio 4), la alarma de la fuga de datos, la evaluación final en el test (ejercicio 5), la parte opcional con MLflow, el mini-reto para tu proyecto y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2 y 3 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, time, json, random, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid")
print("Listo ✔")

### ✏️ A2 · EJERCICIO 1 de 3 — Fijar las semillas

**Tu turno:** Escribe `fijar_semillas(semilla)` que fije la semilla de `random` y de `numpy` (`np.random.seed`), guarde `PYTHONHASHSEED` en `os.environ` (como texto) y **devuelva** un generador `np.random.default_rng(semilla)`. (Con TensorFlow añadirías `tf.random.set_seed(semilla)`.)

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A2).

1. ¿Por qué hay que fijar una semilla en cada biblioteca?
2. ¿Por qué `PYTHONHASHSEED` se guarda con `str(semilla)`?
3. ¿Qué devuelve `fijar_semillas`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A2, **después de intentarlo**.

In [ ]:
def fijar_semillas(semilla):
    # pista: cada biblioteca con azar tiene su propia semilla; la variable de entorno debe ser un str; default_rng crea el generador moderno de numpy
    ...

SEMILLA = 42
rng = fijar_semillas(SEMILLA)
a = (random.random(), np.random.rand(), rng.random())
rng = fijar_semillas(SEMILLA)
b = (random.random(), np.random.rand(), rng.random())
assert a == b, "Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia aleatoria en random, np.random y el rng devuelto"
assert os.environ.get("PYTHONHASHSEED") == "42", "PYTHONHASHSEED debe guardarse como texto '42'"
print("✔ Ejercicio 1 correcto")

### 👀 A3 · EJEMPLO — La ficha del entorno

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Por qué anotar las versiones de las librerías?
2. ¿Dónde conviene copiar esta ficha?

In [ ]:
# Ficha del entorno: cópiala en la memoria técnica y en el README del equipo
ENTORNO = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
print(json.dumps(ENTORNO, indent=2))

### ✏️ A4 · EJERCICIO 2 de 3 — Escribir una fila en el registro

**Tu turno:** Completa `registrar(...)`, que añade **una fila** al CSV `RUTA_REGISTRO`: si el archivo no existe, lo crea con la cabecera; si existe, añade la fila sin repetir la cabecera. El `id` es el número de filas ya existentes + 1. Devuelve el `id` asignado.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Cómo se calcula el `id` de cada fila?
2. ¿Qué hace `header=not existe`?
3. ¿Qué ventajas tiene un CSV como registro?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    # pista: cuenta las filas con pd.read_csv si el archivo existe; crea un DataFrame de una fila y guárdalo con to_csv en modo "a" (header solo si es nuevo)
    ...

id1 = registrar("prueba", "ModeloDePrueba", {"a": 1}, "abc123", 42, "AUC", 0.5, 0.01, 0.1, "fila de prueba")
id2 = registrar("prueba", "ModeloDePrueba", {"a": 2}, "abc123", 42, "AUC", 0.6, 0.01, 0.1)
tabla_prueba = pd.read_csv(RUTA_REGISTRO)
assert (id1, id2) == (1, 2), "Los id deben ser 1, 2, ... (filas existentes + 1)"
assert list(tabla_prueba.columns) == COLUMNAS and len(tabla_prueba) == 2, "El CSV debe tener la cabecera una sola vez y 2 filas"
os.remove(RUTA_REGISTRO)              # borramos las filas de prueba
print("✔ Ejercicio 2 correcto")

### 👀 A5 · EJEMPLO — Preparar los datos y la función experimento (baseline y regresión logística)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Qué AUC tiene el baseline y qué significa?
2. ¿Qué AUC medio logra la regresión logística y con qué desviación?
3. ¿Para qué sirve la huella de los datos (`DATOS_HASH`)?

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

# huella de los datos: si alguien cambia el CSV, el hash cambia y lo sabremos
DATOS_HASH = hashlib.md5(pd.util.hash_pandas_object(reservas, index=True).values.tobytes()).hexdigest()[:8]
print("Huella de los datos:", DATOS_HASH, "· entrenamiento:", X_ent.shape, "· test (sin tocar):", X_test.shape)

categoricas = X.select_dtypes("object").columns.tolist()
def crear_pipeline(estimador, columnas):
    numericas = [c for c in columnas if c not in categoricas]
    prep = ColumnTransformer([("num", StandardScaler(), numericas), ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])
    return Pipeline([("prep", prep), ("m", estimador)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def experimento(nombre, estimador, hiperparametros, responsable="equipo", notas="", Xd=None):
    """Evalúa con CV (AUC), mide el tiempo y registra la fila. Devuelve (media, desviación)."""
    Xd = X_ent if Xd is None else Xd
    t0 = time.time()
    puntuaciones = cross_val_score(crear_pipeline(estimador, Xd.columns), Xd, y_ent, cv=cv, scoring="roc_auc")
    registrar(responsable, nombre, hiperparametros, DATOS_HASH, SEMILLA, "AUC", puntuaciones.mean(), puntuaciones.std(), time.time() - t0, notas)
    return puntuaciones.mean(), puntuaciones.std()

print("Baseline (predice la clase más frecuente):", experimento("Baseline", DummyClassifier(strategy="prior"), {"strategy": "prior"}, notas="referencia mínima"))
print("Regresión logística:", experimento("RegresionLogistica", LogisticRegression(max_iter=2000), {"C": 1.0}))

### ✏️ A6 · EJERCICIO 3 de 3 — Registrar bosque y gradient boosting y elegir candidato

**Tu turno:** Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`: un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)` con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

**Debe salir:** `candidato provisional: RegresionLogistica`, una tabla con 4 filas y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Qué modelo sale como candidato provisional?
2. ¿Qué AUC medio tienen el bosque y el gradient boosting?
3. ¿Qué más, además del AUC, se mira para elegir?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: llama a experimento dos veces (RandomForest y HistGradientBoosting); lee el CSV con RUTA_REGISTRO; idxmax sobre valor_cv te da la fila del mejor
tabla = ...
candidato = ...

assert len(tabla) == 4, "El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting"
assert set(["RandomForest", "HistGradientBoosting"]) <= set(tabla["modelo"]), "Registra los dos modelos con los nombres indicados"
assert candidato == tabla.sort_values("valor_cv").iloc[-1]["modelo"], "candidato debe ser el modelo con mayor valor_cv"
print("candidato provisional:", candidato)
tabla[["id", "modelo", "hiperparametros", "valor_cv", "desv_cv", "tiempo_s"]]
print("✔ Ejercicio 3 correcto")

### 👀 A7 · EJEMPLO — Comparativa de modelos en validación cruzada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué modelo tiene la barra más larga?
2. ¿Qué indica la barra de error de cada modelo?
3. Si dos modelos difieren menos que su desviación, ¿cuál eliges?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
t = tabla.sort_values("valor_cv")
ax.barh(t["modelo"], t["valor_cv"], xerr=t["desv_cv"], color="tab:blue", capsize=4)
ax.axvline(0.5, color="gray", linestyle="--", label="Azar (AUC 0,5)")
ax.set_xlim(0.4, 0.85); ax.set_xlabel("AUC medio en validación cruzada (± desviación)")
ax.set_title("Comparativa de modelos (validación cruzada, sin usar el test)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/para_casa/S15_01_seguimiento_experimentos.ipynb)** trae el ajuste de hiperparámetros registrado, la alarma de la fuga de datos, la evaluación final en el test, MLflow, el mini-reto para tu proyecto y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** En el proyecto final tu equipo probará muchos modelos y muchos ajustes. Al tercer día nadie recuerda «¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?». La rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier equipo, con las reservas de TurisData como datos de práctica.

- **Qué hace:**
    - Carga las herramientas generales (`os`, `sys`, `time`, `json`, `random`, `hashlib`, `datetime`), `numpy`, `pandas`, `matplotlib`, `seaborn` y, de `scikit-learn`, la validación cruzada, el preprocesado, los modelos (baseline, regresión logística, bosque aleatorio y gradient boosting) y el AUC.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Reproducibilidad: semillas y versiones

#### A2 · Ejercicio 1 — Fijar las semillas

**Idea.** Muchos pasos usan azar (separar datos, inicializar pesos, submuestrear). Si no fijas la
**semilla**, cada ejecución da un resultado ligeramente distinto. Además, distintas **versiones** de las
librerías pueden cambiar resultados. Regla del equipo: **una función `fijar_semillas` al inicio de cada
notebook** y las versiones anotadas en el registro.

- **Qué hace:**
    - `random.seed(semilla)` y `np.random.seed(semilla)` fijan la semilla de cada biblioteca con azar.
    - `os.environ["PYTHONHASHSEED"] = str(semilla)` guarda la semilla como **texto** en una variable de entorno (Python solo admite cadenas ahí).
    - `np.random.default_rng(semilla)` crea el generador moderno de NumPy y se **devuelve**.
    - La comprobación llama dos veces a la función con 42 y verifica que `random`, `np.random` y el generador devuelven los mismos números.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto` (la función no imprime nada).
- **Cómo leerlo:**
    - Cada librería tiene su propio generador: hay que fijarlos **todos**. Con TensorFlow se añadiría `tf.random.set_seed(semilla)`. Una función al inicio del cuaderno basta para que otra persona obtenga *exactamente* tus resultados.
- **Si falla:**
    - `Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia…` → falta fijar alguna de las tres semillas (`random`, `np.random`) o no devuelves `default_rng(semilla)`.
    - `PYTHONHASHSEED debe guardarse como texto '42'` → has guardado el número, no el texto (usa `str(semilla)`).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué hay que fijar una semilla en cada biblioteca?* → Porque `random`, NumPy y los modelos tienen **generadores independientes**: fijar uno no fija los demás.
2. *¿Por qué `PYTHONHASHSEED` se guarda con `str(semilla)`?* → Porque las variables de entorno solo admiten **texto**.
3. *¿Qué devuelve `fijar_semillas`?* → Un **generador** `np.random.default_rng(semilla)` para usarlo después.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def fijar_semillas(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    os.environ["PYTHONHASHSEED"] = str(semilla)
    return np.random.default_rng(semilla)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A3 · Ejemplo — La ficha del entorno

- **Qué hace:**
    - `ENTORNO` guarda las versiones de Python, NumPy, pandas y scikit-learn; `json.dumps(..., indent=2)` las imprime con sangría.
- **Qué debe salir:**
    - Un diccionario con cuatro claves: `python`, `numpy`, `pandas` y `scikit-learn`, cada una con su versión (las de tu entorno; por ejemplo `"scikit-learn": "1.9.1"`).
- **Cómo leerlo:**
    - Distintas versiones de una librería pueden cambiar los resultados. Copia esta ficha en la memoria técnica y en el README del equipo.
- **Si falla:**
    - `NameError: name 'sklearn' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué anotar las versiones de las librerías?* → Porque **otra versión puede dar otros resultados**: sin ellas, no se puede reproducir con certeza.
2. *¿Dónde conviene copiar esta ficha?* → En la **memoria técnica** y en el **README** del equipo.

</details>

## 2. El registro de experimentos: una fila por prueba

#### A4 · Ejercicio 2 — Escribir una fila en el registro

**Idea.** Cada vez que entrenas y evalúas, anotas una fila: **cuándo, qué modelo, con qué ajustes, con
qué datos, qué métrica y cuánto tardó**. Un CSV en el repositorio del equipo (`experimentos/registro.csv`)
se puede abrir con Excel, se versiona con Git y no requiere instalar nada.

Columnas mínimas: `id`, `fecha`, `responsable`, `modelo`, `hiperparametros` (JSON), `datos_hash`,
`semilla`, `metrica`, `valor_cv`, `desv_cv`, `tiempo_s`, `notas`.

- **Qué hace:**
    - Las primeras líneas (ya hechas) definen la ruta del registro, la lista de columnas, crean la carpeta `experimentos` y **vacían el registro** para empezar de cero.
    - `registrar(...)` mira si el CSV existe (`existe`) y cuenta sus filas (`n`); el `id` es `n + 1`.
    - Crea un `DataFrame` de una fila con todos los campos (la fecha actual, los hiperparámetros como JSON, los números redondeados) y lo añade con `to_csv(..., mode="a", header=not existe, index=False)`: añade sin repetir la cabecera.
    - La comprobación registra dos filas de prueba, verifica los `id` y las columnas, y borra el archivo.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto` (la función no imprime nada).
- **Cómo leerlo:**
    - El registro es un **CSV**: se abre con Excel, se versiona con Git y no necesita instalar nada. Cada llamada añade una fila con cuándo, quién, qué modelo, con qué ajustes y datos, qué métrica y cuánto tardó.
- **Si falla:**
    - `Los id deben ser 1, 2, ... (filas existentes + 1)` → no cuentas las filas existentes o devuelves otro valor.
    - `El CSV debe tener la cabecera una sola vez y 2 filas` → escribes la cabecera en cada llamada (`header` debe ser `True` solo si el archivo es nuevo) o no usas el modo `"a"` (añadir).
    - `NameError: name 'datetime' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cómo se calcula el `id` de cada fila?* → **Filas ya existentes + 1**.
2. *¿Qué hace `header=not existe`?* → Escribe la cabecera **solo la primera vez** (cuando el archivo no existía).
3. *¿Qué ventajas tiene un CSV como registro?* → Se abre con **Excel**, se **versiona con Git** y no requiere instalar nada.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    existe = os.path.exists(RUTA_REGISTRO)
    n = len(pd.read_csv(RUTA_REGISTRO)) if existe else 0
    fila = pd.DataFrame([{
        "id": n + 1, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"), "responsable": responsable, "modelo": modelo,
        "hiperparametros": json.dumps(hiperparametros, sort_keys=True), "datos_hash": datos_hash, "semilla": semilla,
        "metrica": metrica, "valor_cv": round(float(valor_cv), 4), "desv_cv": round(float(desv_cv), 4),
        "tiempo_s": round(float(tiempo_s), 2), "notas": notas}], columns=COLUMNAS)
    fila.to_csv(RUTA_REGISTRO, mode="a", header=not existe, index=False)
    return n + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Baseline y modelos con validación cruzada

#### A5 · Ejemplo — Preparar los datos y la función experimento (baseline y regresión logística)

**Idea.** Para comparar modelos se usa **validación cruzada**: se parte el conjunto de entrenamiento en 5
trozos y se entrena 5 veces, evaluando cada vez en el trozo que no se ha visto. Obtenemos una **media**
y una **desviación** (¡importante para saber si una diferencia es real o ruido!). El **test** se
reserva para el final y se usa **una sola vez**. Con series temporales, la partición debe respetar el orden
del tiempo (`TimeSeriesSplit`), nunca al azar.

**Ejemplo resuelto:** preparamos datos y una función que entrena, evalúa y **registra automáticamente**.

- **Qué hace:**
    - Carga las reservas, separa `X` e `y` y guarda el 20 % para el test (`X_test`, que **no se toca** hasta el final).
    - `DATOS_HASH` es la huella de los datos: si alguien cambia el CSV, cambia.
    - `crear_pipeline` prepara los datos y el modelo en una sola pieza; `cv` es la validación cruzada de 5 trozos.
    - `experimento(...)` evalúa un modelo con validación cruzada (AUC), mide el tiempo, **registra la fila** con `registrar` y devuelve la media y la desviación.
    - Se prueban el baseline (predice la clase más frecuente) y la regresión logística.
- **Qué debe salir:**
    - `Huella de los datos: 800f40b2 · entrenamiento: (4800, 15) · test (sin tocar): (1200, 15)`, `Baseline (predice la clase más frecuente): (0.5, 0.0)` y `Regresión logística: (0.755, 0.0124)` (según la versión de NumPy pueden aparecer como `np.float64(0.5)`).
- **Cómo leerlo:**
    - El **baseline** da AUC 0,5 (el azar): es la referencia mínima. La logística consigue **0,755 ± 0,012**. La desviación importa: dice cuánto cambia el resultado según el trozo.
- **Si falla:**
    - `NameError: name 'registrar' is not defined` → falta el ejercicio 2.
    - `NameError: name 'SEMILLA' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tiene el baseline y qué significa?* → **0,5**: es el azar, la referencia mínima que cualquier modelo debe superar.
2. *¿Qué AUC medio logra la regresión logística y con qué desviación?* → **0,755** ± **0,012**.
3. *¿Para qué sirve la huella de los datos (`DATOS_HASH`)?* → Para saber si **alguien ha cambiado el CSV**: si cambia, la huella cambia.

</details>

#### A6 · Ejercicio 3 — Registrar bosque y gradient boosting y elegir candidato

Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`:
un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)`
con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con
nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro
en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

- **Qué hace:**
    - `experimento("RandomForest", RandomForestClassifier(...), {...})` y `experimento("HistGradientBoosting", HistGradientBoostingClassifier(...), {...})` evalúan los dos modelos con validación cruzada y **registran** una fila cada uno.
    - `pd.read_csv(RUTA_REGISTRO)` lee el registro (`tabla`).
    - `tabla.loc[tabla["valor_cv"].idxmax(), "modelo"]` es el nombre del modelo con mayor AUC medio (`candidato`).
- **Qué debe salir:**
    - `candidato provisional: RegresionLogistica`, una tabla con 4 filas (`Baseline` 0.5000, `RegresionLogistica` 0.7550, `RandomForest` 0.7393 y `HistGradientBoosting` 0.7275) y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - La logística gana, con el bosque y el gradient boosting algo por debajo. Pero fíjate en las desviaciones (0,011–0,013): las diferencias entre modelos son pequeñas. Y el bosque tarda unos 3,6 s frente a 0,14 s de la logística: el tiempo también es un criterio.
- **Si falla:**
    - `El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting` → falta registrar alguno de los dos modelos (o los has registrado dos veces).
    - `Registra los dos modelos con los nombres indicados` → los nombres deben ser exactamente `"RandomForest"` y `"HistGradientBoosting"`.
    - `candidato debe ser el modelo con mayor valor_cv` → no has usado `idxmax` sobre `valor_cv`.
    - `NameError: name 'experimento' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo sale como candidato provisional?* → **RegresionLogistica**, con AUC medio **0,755**.
2. *¿Qué AUC medio tienen el bosque y el gradient boosting?* → **0,739** y **0,728**.
3. *¿Qué más, además del AUC, se mira para elegir?* → La **desviación** (si la diferencia es real) y el **tiempo** de cada modelo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
experimento("RandomForest", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2),
            {"n_estimators": 200, "min_samples_leaf": 5})
experimento("HistGradientBoosting", HistGradientBoostingClassifier(random_state=SEMILLA), {"defecto": True})
tabla = pd.read_csv(RUTA_REGISTRO)
candidato = tabla.loc[tabla["valor_cv"].idxmax(), "modelo"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A7 · Ejemplo — Comparativa de modelos en validación cruzada

- **Qué hace:**
    - Ordena la tabla por `valor_cv` y dibuja barras horizontales con el AUC medio de cada modelo y su desviación como barra de error (`xerr`).
    - La línea discontinua marca el azar (AUC 0,5).
- **Qué debe salir:**
    - Un gráfico «Comparativa de modelos (validación cruzada, sin usar el test)» con cuatro barras: `Baseline` en 0,5 y las demás entre 0,73 y 0,76, con barras de error pequeñas.
- **Cómo leerlo:**
    - si dos modelos difieren menos que su desviación, **no hay diferencia real**.
En ese caso elige el más simple, más rápido o más fácil de explicar. Anótalo en la columna `notas`.
- **Si falla:**
    - `NameError: name 'tabla' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué modelo tiene la barra más larga?* → La **regresión logística** (AUC ≈ 0,755).
2. *¿Qué indica la barra de error de cada modelo?* → La **desviación** de la validación cruzada: cuánto varía el resultado entre trozos.
3. *Si dos modelos difieren menos que su desviación, ¿cuál eliges?* → El **más simple, rápido o fácil de explicar**: no hay diferencia real.

</details>